# Layer 4: RL Agent Training (PPO per Regime)

This notebook trains three separate PPO policies (one per market regime) for
regime-conditional portfolio rebalancing using the **RAPO-AS-RL** framework.

## Environment Design

**State Space** (11 dimensions):
```
[w_btc, w_eth, w_cash, regime_idx, mu_btc, mu_eth, sigma, spread, depth, sigma_port, cum_pnl]
```

- `w_btc, w_eth, w_cash`: Current portfolio weights (BTC, ETH, cash)
- `regime_idx`: HMM regime index (0=Calm, 1=Volatile, 2=Stressed)
- `mu_btc, mu_eth`: LightGBM forecasted returns for each asset
- `sigma`: Per-regime A&S volatility parameter
- `spread`: Per-regime A&S spread parameter
- `depth`: Per-regime A&S market depth parameter
- `sigma_port`: Rolling 20-period portfolio volatility
- `cum_pnl`: Cumulative realized PnL

**Action Space** (2 dimensions, continuous):
```
[target_w_btc, target_w_eth]  →  w_cash = 1 - w_btc - w_eth
```
Actions are normalized to the simplex to ensure valid weight constraints.

**Reward Function**:
```
r_t = (portfolio_return - A&S_cost) / sigma_port
```

Episode: One 5-min bar. Regime transitions handled by HMM at each step.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Change to project root
%cd C:/Users/zihan/capstone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from src.layer4_rl.rl_env import (
    RegimePortfolioEnv,
    create_synthetic_forecasters,
    create_synthetic_cost_models,
    create_synthetic_price_data,
    create_synthetic_regime_labels,
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

DATA_DIR = Path("data/processed")
MODELS_DIR = Path("models")
RL_MODELS_DIR = MODELS_DIR / "rl"
RL_MODELS_DIR.mkdir(parents=True, exist_ok=True)

REGIMES = ["Calm", "Volatile", "Stressed"]
ASSETS = ["BTC", "ETH"]

## 1. Load Data or Create Synthetic Fallback

In [ ]:
# Use REAL data - all prerequisite models (HMM, A&S, LightGBM) are available
use_synthetic = False

if use_synthetic:
    print("⚠️  Using SYNTHETIC data (Layer 3 LightGBM models not available):")
    print("   This is expected during development/testing.")
else:
    print("✓ All prerequisite files found! Using real data.")
    print(f"   - A&S cost models: models/as_cost/as_cost_{{calm,volatile,stressed}}.pkl")
    print(f"   - LightGBM forecasters: models/lgbm/lgbm_{{btc,eth}}_{{calm,volatile}}.pkl")
    print(f"   - HMM regime labels: models/hmm/regime_labels.csv")

In [ ]:
if use_synthetic:
    # Generate synthetic data for testing
    print("Generating synthetic data...")
    price_df = create_synthetic_price_data(n_bars=10000, freq='5min')
    regime_labels = create_synthetic_regime_labels(len(price_df), price_df)
    as_cost_models = create_synthetic_cost_models()
    forecasters = create_synthetic_forecasters()
    print(f"Synthetic data shape: {price_df.shape}")
    print(f"Date range: {price_df.index.min()} to {price_df.index.max()}")
    print(f"\nRegime distribution:")
    print(regime_labels.value_counts())
else:
    # Load real data
    print("Loading real data...")
    price_df = pd.read_parquet(DATA_DIR / "price_features.parquet")
    print(f"Price data shape: {price_df.shape}")
    print(f"Date range: {price_df.index.min()} to {price_df.index.max()}")
    
    # Load HMM regime labels (FIX: squeeze deprecated in newer pandas)
    regime_labels = pd.read_csv(MODELS_DIR / "hmm" / "regime_labels.csv", index_col=0).iloc[:, 0]
    regime_labels.index = pd.to_datetime(regime_labels.index)
    print(f"Regime distribution:")
    print(regime_labels.value_counts())
    
    # Load A&S cost models
    as_cost_models = {}
    for regime in REGIMES:
        pkl_path = MODELS_DIR / "as_cost" / f"as_cost_{regime.lower()}.pkl"
        if pkl_path.exists():
            as_cost_models[regime] = joblib.load(pkl_path)
        else:
            print(f"Warning: A&S model not found for {regime}, using synthetic")
            as_cost_models[regime] = create_synthetic_cost_models()[regime]
    
    # Load LightGBM forecasters
    forecasters = {asset: {} for asset in ASSETS}
    for asset in ASSETS:
        for regime in REGIMES:
            pkl_path = MODELS_DIR / "lgbm" / f"lgbm_{asset.lower()}_{regime.lower()}.pkl"
            if pkl_path.exists():
                forecasters[asset][regime] = joblib.load(pkl_path)
            else:
                print(f"Warning: LightGBM not found for {asset}/{regime}, using synthetic")
                forecasters[asset][regime] = create_synthetic_forecasters()[asset][regime]

In [ ]:
# DELETE: Old redundant cells - data loading merged into cell-4

In [ ]:
# DELETE: Old redundant cells - data loading merged into cell-4

In [ ]:
# DELETE: Old redundant cells - data loading merged into cell-4

## 2. Create RL Environment

In [ ]:
# Create training and evaluation environments
env = RegimePortfolioEnv(
    price_data=price_df,
    regime_labels=regime_labels,
    as_cost_models=as_cost_models,
    forecasters=forecasters,
    initial_balance=100_000.0,
)

eval_env = RegimePortfolioEnv(
    price_data=price_df,
    regime_labels=regime_labels,
    as_cost_models=as_cost_models,
    forecasters=forecasters,
    initial_balance=100_000.0,
)

print(f"Environment created with {len(price_df)} time steps")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"Max episode steps: {env._max_episode_steps}")

In [ ]:
# Test environment with random action
obs, info = env.reset()
print(f"Initial observation shape: {obs.shape}")
print(f"Initial observation: {obs}")

# Take a random action
action = env.action_space.sample()
obs, reward, done, truncated, info = env.step(action)
print(f"\nAfter random action:")
print(f"Reward: {reward:.6f}")
print(f"Done: {done}, Truncated: {truncated}")

## 3. Define Training Functions

In [ ]:
def rolling_sharpe(rewards, window=100):
    """Compute rolling Sharpe ratio."""
    if len(rewards) < window:
        return 0.0
    recent = rewards[-window:]
    return np.mean(recent) / (np.std(recent) + 1e-8)


def evaluate_agent(model, env, n_episodes=10, n_steps=500):
    """Evaluate agent and return metrics."""
    episode_rewards = []
    episode_returns = []
    episode_lengths = []
    
    for _ in range(n_episodes):
        obs, _ = env.reset()
        total_reward = 0
        step_count = 0
        
        while step_count < n_steps:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, truncated, _ = env.step(action)
            total_reward += reward
            step_count += 1
            
            if done or truncated:
                break
        
        episode_rewards.append(total_reward)
        episode_lengths.append(step_count)
    
    return {
        'mean_reward': np.mean(episode_rewards),
        'std_reward': np.std(episode_rewards),
        'mean_length': np.mean(episode_lengths),
    }


def evaluate_with_portfolio_metrics(model, env, n_episodes=5):
    """Evaluate agent with detailed portfolio metrics."""
    all_rewards = []
    all_portfolio_values = []
    all_regimes = []
    all_rebalances = []
    
    for episode in range(n_episodes):
        obs, _ = env.reset()
        rewards = []
        portfolio_values = [env.portfolio_value]
        rebalance_count = 0
        prev_weights = env.current_weights.copy()
        
        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, truncated, _ = env.step(action)
            rewards.append(reward)
            portfolio_values.append(env.portfolio_value)
            
            # Count rebalancing events
            if not np.allclose(env.current_weights[:2], prev_weights[:2], atol=1e-4):
                rebalance_count += 1
            prev_weights = env.current_weights.copy()
            
            regime_idx = int(obs[3])
            all_regimes.append(regime_idx)
            
            if done or truncated:
                break
        
        all_rewards.extend(rewards)
        all_portfolio_values.append(portfolio_values)
        all_rebalances.append(rebalance_count)
    
    # Compute metrics
    returns = []
    for pf_values in all_portfolio_values:
        pf_arr = np.array(pf_values)
        ret = (pf_arr[1:] - pf_arr[:-1]) / pf_arr[:-1]
        returns.extend(ret)
    returns = np.array(returns)
    
    sharpe = np.mean(returns) / (np.std(returns) + 1e-8) * np.sqrt(288 * 365) if len(returns) > 0 else 0
    cumulative = np.cumprod(1 + returns)
    running_max = np.maximum.accumulate(cumulative)
    drawdowns = (cumulative - running_max) / running_max
    max_drawdown = np.min(drawdowns) if len(drawdowns) > 0 else 0
    
    return {
        'sharpe': sharpe,
        'max_drawdown': max_drawdown,
        'mean_return': np.mean(returns),
        'std_return': np.std(returns),
        'mean_rebalances': np.mean(all_rebalances),
    }

In [ ]:
# Import train_regime_ppo from src instead of redefining
# (removes duplication between notebook and src/layer4_rl/rl_train.py)
import importlib
import sys

# Ensure src is on path
sys.path.insert(0, str(Path.cwd() / 'src'))

# Import from src - note: we re-export train_regime_ppo for notebook use
from layer4_rl.rl_train import train_regime_ppo as _train_regime_ppo
from layer4_rl.rl_env import RegimePortfolioEnv

# For notebook convenience, wrap to match expected return signature
def train_regime_ppo(env, regime, n_timesteps=200_000, verbose=1):
    """Wrapper that matches notebook's expected return signature."""
    model, best_params, best_sharpe, all_results = _train_regime_ppo(env, regime, n_timesteps)
    return model, best_params, best_sharpe, all_results

print("train_regime_ppo imported from src.layer4_rl.rl_train")

In [ ]:
# =============================================================================
# Cell 14: Training OR Load Existing Models
# =============================================================================
# If RL model files already exist from a previous run, load them instead of
# retraining. This allows the notebook to be re-executed for visualization
# without overwriting the trained policies.
#
# To force retraining: delete models/rl/ppo_*.zip first

import json
import os

trained_models = {}
training_results = {}
N_TIMESTEPS = 200_000

# Check which models already exist
existing_models = {}
for regime in REGIMES:
    model_path = RL_MODELS_DIR / f"ppo_{regime.lower()}.zip"
    if model_path.exists():
        existing_models[regime] = model_path

if len(existing_models) == len(REGIMES):
    # All models exist - load them instead of retraining
    print("All RL models found on disk. Loading instead of retraining.")
    print(f"  Configs: 2 × {N_TIMESTEPS:,} timesteps per regime")
    
    for regime in REGIMES:
        model_path = existing_models[regime]
        model = PPO.load(str(model_path))
        trained_models[regime] = model
        print(f"  Loaded: {model_path.name}")
    
    # Load training_results.json if available
    tr_path = RL_MODELS_DIR / "training_results.json"
    if tr_path.exists():
        with open(tr_path) as f:
            all_tr = json.load(f)
        for regime in REGIMES:
            training_results[regime] = all_tr.get(regime, {}).get('config_results', [])
        print(f"  Loaded: training_results.json")
else:
    # Some or all models missing - train from scratch
    missing = [r for r in REGIMES if r not in existing_models]
    print(f"Models missing for: {missing}. Training...")
    print(f"Training with N_TIMESTEPS = {N_TIMESTEPS:,} per regime")
    print(f"Configs: lr={{3e-4, 1e-4}} × γ={{0.99}} × clip={{0.2, 0.1}}")
    
    for regime in REGIMES:
        if regime in existing_models:
            model = PPO.load(str(existing_models[regime]))
            trained_models[regime] = model
            print(f"  Loaded existing: {regime}")
            continue
            
        print(f"\n{'='*60}")
        print(f"Training PPO for {regime} regime...")
        print('='*60)
        
        model, params, sharpe, results = train_regime_ppo(
            env, regime, n_timesteps=N_TIMESTEPS, verbose=1
        )
        
        model_path = RL_MODELS_DIR / f"ppo_{regime.lower()}.zip"
        model.save(str(model_path))
        print(f"  Saved: {model_path}")
        
        trained_models[regime] = model
        training_results[regime] = results
    
    # Save training results JSON
    results_json = {}
    for regime in REGIMES:
        results_json[regime] = {
            "best_params": training_results[regime][0]["params"] if training_results[regime] else {},
            "best_sharpe": max(r["final_sharpe"] for r in training_results[regime]) if training_results[regime] else 0,
            "config_results": training_results[regime]
        }
    with open(RL_MODELS_DIR / "training_results.json", "w") as f:
        json.dump(results_json, f, indent=2)
    print(f"\nTraining results saved to {RL_MODELS_DIR / 'training_results.json'}")

In [ ]:
# NOTE: Training was already completed in cell-14.
# This cell intentionally left as a placeholder to avoid breaking notebook flow.
# Do NOT re-run training here - it would overwrite the correct 200K models with 1M training.
print("Training already completed in cell-14. Skipping duplicate training.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, regime in enumerate(REGIMES):
    ax = axes[idx]
    
    results = training_results[regime]
    for result in results:
        label = f"lr={result['params']['learning_rate']}, " \
                f"γ={result['params']['gamma']}, " \
                f"clip={result['params']['clip_range']}"
        ax.plot(result['sharpe_history'], label=label, alpha=0.7)
    
    ax.set_title(f'{regime} Regime - Training Sharpe')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Rolling Sharpe (100-step)')
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RL_MODELS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RL_MODELS_DIR / 'training_curves.png'}")

In [ ]:
print("\nEvaluating trained policies...")
print("="*60)

eval_metrics = {}
for regime in REGIMES:
    if regime in trained_models:
        print(f"\n{regime} Regime:")
        metrics = evaluate_with_portfolio_metrics(trained_models[regime], eval_env, n_episodes=5)
        eval_metrics[regime] = metrics
        print(f"  Sharpe Ratio (annualized): {metrics['sharpe']:.3f}")
        print(f"  Max Drawdown: {metrics['max_drawdown']*100:.2f}%")
        print(f"  Mean Return (per bar): {metrics['mean_return']*100:.4f}%")
        print(f"  Std Return (per bar): {metrics['std_return']*100:.4f}%")
        print(f"  Mean Rebalances/Episode: {metrics['mean_rebalances']:.1f}")

## 6. Evaluate Trained Policies

In [ ]:
if trained_models:
    print("\nEvaluating trained policies...")
    print("="*60)
    
    eval_metrics = {}
    for regime in REGIMES:
        if regime in trained_models:
            print(f"\n{regime} Regime:")
            metrics = evaluate_with_portfolio_metrics(trained_models[regime], eval_env, n_episodes=5)
            eval_metrics[regime] = metrics
            print(f"  Sharpe Ratio: {metrics['sharpe']:.3f}")
            print(f"  Max Drawdown: {metrics['max_drawdown']*100:.2f}%")
            print(f"  Mean Return: {metrics['mean_return']*100:.4f}%")
            print(f"  Std Return: {metrics['std_return']*100:.4f}%")
            print(f"  Mean Rebalances/Episode: {metrics['mean_rebalances']:.1f}")
else:
    print("Evaluation not available (no trained models).")

In [ ]:
print("\n" + "="*60)
print("RL Policy vs Random Baseline Comparison")
print("="*60)

comparison_results = {}

for regime in REGIMES:
    print(f"\n{regime} Regime:")
    
    # RL Policy metrics
    rl_metrics = eval_metrics[regime]
    
    # Random baseline metrics
    random_policy = RandomPolicy(env.action_space)
    random_metrics = evaluate_with_portfolio_metrics(random_policy, eval_env, n_episodes=5)
    
    print(f"  RL Policy:")
    print(f"    Sharpe: {rl_metrics['sharpe']:.3f}, " \
          f"Max DD: {rl_metrics['max_drawdown']*100:.2f}%, " \
          f"Rebalances: {rl_metrics['mean_rebalances']:.1f}")
    print(f"  Random Baseline:")
    print(f"    Sharpe: {random_metrics['sharpe']:.3f}, " \
          f"Max DD: {random_metrics['max_drawdown']*100:.2f}%, " \
          f"Rebalances: {random_metrics['mean_rebalances']:.1f}")
    
    # Calculate improvement
    if abs(random_metrics['sharpe']) > 1e-8:
        sharpe_improvement = ((rl_metrics['sharpe'] - random_metrics['sharpe']) / abs(random_metrics['sharpe'])) * 100
    else:
        sharpe_improvement = float('inf') if rl_metrics['sharpe'] > 0 else float('-inf')
    
    print(f"  Sharpe Improvement: {sharpe_improvement:.1f}%")
    print(f"  RL Outperforms Random: {'YES' if rl_metrics['sharpe'] > random_metrics['sharpe'] else 'NO'}")
    
    comparison_results[regime] = {
        'rl': rl_metrics,
        'random': random_metrics,
    }

In [ ]:
class RandomPolicy:
    """Random baseline policy for comparison."""
    def __init__(self, action_space):
        self.action_space = action_space
    def predict(self, obs, deterministic=False):
        return self.action_space.sample(), None

if trained_models and eval_metrics:
    print("\n" + "="*60)
    print("RL Policy vs Random Baseline Comparison")
    print("="*60)
    
    comparison_results = {}
    
    for regime in REGIMES:
        if regime not in eval_metrics:
            continue
        print(f"\n{regime} Regime:")
        
        rl_metrics = eval_metrics[regime]
        random_policy = RandomPolicy(env.action_space)
        random_metrics = evaluate_with_portfolio_metrics(random_policy, eval_env, n_episodes=5)
        
        print(f"  RL Policy:")
        print(f"    Sharpe: {rl_metrics['sharpe']:.3f}, "
              f"Max DD: {rl_metrics['max_drawdown']*100:.2f}%, "
              f"Rebalances: {rl_metrics['mean_rebalances']:.1f}")
        print(f"  Random Baseline:")
        print(f"    Sharpe: {random_metrics['sharpe']:.3f}, "
              f"Max DD: {random_metrics['max_drawdown']*100:.2f}%, "
              f"Rebalances: {random_metrics['mean_rebalances']:.1f}")
        print(f"  Improvement:")
        print(f"    Sharpe: {((rl_metrics['sharpe'] - random_metrics['sharpe']) / (abs(random_metrics['sharpe']) + 1e-8) * 100):.1f}%")
        
        comparison_results[regime] = {'rl': rl_metrics, 'random': random_metrics}
else:
    print("Comparison not available (no trained models or eval metrics).")

In [ ]:
print("\n" + "="*60)
print("Regime-Adaptive Rebalancing Behavior")
print("="*60)

rebalance_by_regime = {}

for regime in REGIMES:
    # Evaluate on same regime multiple times
    rebalances = []
    for _ in range(3):
        obs, _ = eval_env.reset()
        rebalance_count = 0
        prev_weights = eval_env.current_weights.copy()
        
        while True:
            action, _ = trained_models[regime].predict(obs, deterministic=True)
            obs, reward, done, truncated, _ = eval_env.step(action)
            
            if not np.allclose(eval_env.current_weights[:2], prev_weights[:2], atol=1e-4):
                rebalance_count += 1
            prev_weights = eval_env.current_weights.copy()
            
            if done or truncated:
                break
        
        rebalances.append(rebalance_count)
    
    rebalance_by_regime[regime] = rebalances
    print(f"{regime} Regime: {np.mean(rebalances):.1f} ± {np.std(rebalances):.1f} rebalances per episode")

print("\nInterpretation:")
print("- Calm: Typically lower rebalancing (trend-following, less noise)")
print("- Volatile: Moderate rebalancing (responding to regime shifts)")
print("- Stressed: Potentially different rebalancing patterns (risk management focus)")

In [ ]:
if trained_models:
    print("\n" + "="*60)
    print("Regime-Adaptive Rebalancing Behavior")
    print("="*60)
    
    rebalance_by_regime = {}
    
    for regime in REGIMES:
        rebalances = []
        for _ in range(3):
            obs, _ = eval_env.reset()
            rebalance_count = 0
            prev_weights = eval_env.current_weights.copy()
            
            while True:
                action, _ = trained_models[regime].predict(obs, deterministic=True)
                obs, reward, done, truncated, _ = eval_env.step(action)
                
                if not np.allclose(eval_env.current_weights[:2], prev_weights[:2], atol=1e-4):
                    rebalance_count += 1
                prev_weights = eval_env.current_weights.copy()
                
                if done or truncated:
                    break
            
            rebalances.append(rebalance_count)
        
        rebalance_by_regime[regime] = rebalances
        print(f"{regime} Regime: {np.mean(rebalances):.1f} ± {np.std(rebalances):.1f} rebalances per episode")
    
    print("\nInterpretation:")
    print("- Calm: Typically lower rebalancing (trend-following, less noise)")
    print("- Volatile: Moderate rebalancing (responding to regime shifts)")
    print("- Stressed: Potentially different rebalancing patterns (risk management focus)")
else:
    print("Regime behavior analysis not available (no trained models).")

In [ ]:
summary_data = []

for regime in REGIMES:
    rl = comparison_results[regime]['rl']
    rnd = comparison_results[regime]['random']
    
    if abs(rnd['sharpe']) > 1e-8:
        sharpe_imp = f"{((rl['sharpe'] - rnd['sharpe']) / abs(rnd['sharpe']) * 100):.1f}%"
    else:
        sharpe_imp = "N/A"
    
    summary_data.append({
        'Regime': regime,
        'RL Sharpe': f"{rl['sharpe']:.3f}",
        'Random Sharpe': f"{rnd['sharpe']:.3f}",
        'RL Max DD': f"{rl['max_drawdown']*100:.2f}%",
        'Random Max DD': f"{rnd['max_drawdown']*100:.2f}%",
        'RL Rebalances': f"{rl['mean_rebalances']:.1f}",
        'Random Rebalances': f"{rnd['mean_rebalances']:.1f}",
        'Sharpe Improvement': sharpe_imp,
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

# Save summary
summary_df.to_csv(RL_MODELS_DIR / 'rl_evaluation_summary.csv', index=False)
print(f"\nSaved: {RL_MODELS_DIR / 'rl_evaluation_summary.csv'}")

In [ ]:
if comparison_results:
    summary_data = []
    
    for regime in REGIMES:
        if regime not in comparison_results:
            continue
        rl = comparison_results[regime]['rl']
        rnd = comparison_results[regime]['random']
        
        summary_data.append({
            'Regime': regime,
            'RL Sharpe': f"{rl['sharpe']:.3f}",
            'Random Sharpe': f"{rnd['sharpe']:.3f}",
            'RL Max DD': f"{rl['max_drawdown']*100:.2f}%",
            'Random Max DD': f"{rnd['max_drawdown']*100:.2f}%",
            'RL Rebalances': f"{rl['mean_rebalances']:.1f}",
            'Random Rebalances': f"{rnd['mean_rebalances']:.1f}",
            'Sharpe Improvement': f"{((rl['sharpe'] - rnd['sharpe']) / (abs(rnd['sharpe']) + 1e-8) * 100):.1f}%",
        })
    
    summary_df = pd.DataFrame(summary_data)
    print("\n" + summary_df.to_string(index=False))
    
    summary_df.to_csv(RL_MODELS_DIR / 'rl_evaluation_summary.csv', index=False)
    print(f"\nSaved: {RL_MODELS_DIR / 'rl_evaluation_summary.csv'}")
else:
    print("Summary not available (no comparison results).")

In [ ]:
print("\n" + "="*60)
print("FINAL DELIVERABLES")
print("="*60)

print("\nSaved RL Models:")
for regime in REGIMES:
    model_path = RL_MODELS_DIR / f"ppo_{regime.lower()}.zip"
    if model_path.exists():
        print(f"  ✓ {model_path}")

print("\nSaved Evaluation Files:")
for f in ['training_curves.png', 'rl_evaluation_summary.csv']:
    fp = RL_MODELS_DIR / f
    if fp.exists():
        print(f"  ✓ {fp}")

print("\nEvidence: RL vs Random Baseline:")
rl_outperforms_all = True
for regime in REGIMES:
    rl_sharpe = comparison_results[regime]['rl']['sharpe']
    rnd_sharpe = comparison_results[regime]['random']['sharpe']
    outperformance = rl_sharpe > rnd_sharpe
    symbol = "✓" if outperformance else "✗"
    print(f"  {symbol} {regime}: RL {rl_sharpe:.3f} vs Random {rnd_sharpe:.3f}")
    if not outperformance:
        rl_outperforms_all = False

print(f"\nRL Policy Outperforms Random in All Regimes: {'YES' if rl_outperforms_all else 'NO'}")

if use_synthetic:
    print("\nNOTE: Results computed using SYNTHETIC data.")
    print("For production results, run with real data from Layers 0-3.")

In [ ]:
# DELETE: Redundant cell - merged into Final Deliverables